# Práctica 1: Neuronas, perceptrón y activaciones

**Fundamentos matemáticos del aprendizaje profundo** · Departamento de Matemática, FCEN-UBA

Material de referencia: capítulos 1, 2 y 5 de las notas del curso. Convenciones: $y=\varphi(w\cdot x+b)$, $\sigma$ es la logística, $\eta$ la tasa de aprendizaje, $w_{ij}$ conecta la entrada $i$ con la salida $j$.

Cada práctica tiene una **Parte A** (ejercicios teóricos, para resolver en papel) y una **Parte B** (computacional). Las celdas marcadas `# TODO` son para completar.

## Parte A. Ejercicios teóricos

**A1.** Una fábrica empieza a producir cuando el suministro diario alcanza una cantidad crítica $b$. Escribir la fórmula de los ingresos diarios $y$ como función del suministro $x$ usando la función de Heaviside $H$ (con $H(0)=1$) y formular el problema de aprendizaje asociado.

**A2.** (a) Mostrar que $0<\sigma'(x)\le\tfrac14$ para $x\in\mathbb R$. (b) Para $\sigma_c(x)=\sigma(cx)$, ¿cuál es la cota correspondiente? ¿Qué pasa con $\sigma_c$ cuando $c\to\infty$?

**A3.** Verificar que $S(x)=2H(x)-1$ y $\mathrm{ReLU}(x)=\tfrac12x\,(S(x)+1)$ son las expresiones correctas de signo y ReLU, y que $\mathrm{ReLU}'=H$ salvo en $0$.

**A4.** Un perceptrón con pesos $w$ y sesgo $b$ dispara ($y=1$) si y solo si $w\cdot x\ge -b$. Mostrar que un perceptrón puede aprender: (a) $x_1\wedge\neg x_2$; (b) $x_1\vee\neg x_2$; (c) $\neg x$ (¿y una neurona lineal?); (d) $x_1\wedge x_2\wedge x_3$ y $x_1\vee x_2\vee x_3$.

**A5.** Mostrar que ningún perceptrón aprende la función XOR. Dar una red de perceptrones con una capa oculta que la aprenda, indicando todos los pesos y sesgos.

**A6.** (Novikoff) Sean datos linealmente separables con $\|x_i\|\le R$ y margen $\gamma>0$ (existe $w^*$ unitario con $y_i\,w^*\cdot x_i\ge\gamma$, $y_i\in\{\pm1\}$). Mostrar que el algoritmo del perceptrón comete a lo sumo $(R/\gamma)^2$ errores.

## Parte B. Computacional

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize']=(6,4)
rng=np.random.default_rng(0)

### B1. Funciones de activación y derivadas
Implementar (vectorizadas) `H`, `sigma`, `relu`, `tanh` y sus derivadas. Graficarlas y verificar numéricamente que $\max\sigma'=1/4$ y $\max\sigma_c'=c/4$.

In [ ]:
H=lambda x: (x>=0).astype(float)
sigma=lambda x: 1/(1+np.exp(-x))
dsigma=lambda x: sigma(x)*(1-sigma(x))
relu=lambda x: np.maximum(x,0)
x=np.linspace(-6,6,601)
fig,ax=plt.subplots(1,2,figsize=(10,3.5))
for nombre,f in [('H',H),('sigma',sigma),('relu',relu),('tanh',np.tanh)]:
    ax[0].plot(x,f(x),label=nombre)
ax[0].legend(); ax[0].set_ylim(-1.5,3)
for c in [1,2,5]:
    ax[1].plot(x,c*dsigma(c*x),label=f'c={c}')
    print(c, (c*dsigma(c*x)).max(), c/4)
ax[1].legend(); ax[1].set_title("derivada de sigma(cx)")
plt.show()

**B1.1** Comparar `dsigma` con una derivada por diferencias centradas $\frac{f(x+h)-f(x-h)}{2h}$ para $h=10^{-1},\dots,10^{-8}$. ¿Para qué $h$ el error es mínimo? ¿Por qué empeora para $h$ muy chico?

In [ ]:
# TODO: error de las diferencias finitas en x0=0.7 en funcion de h (escala log-log)

### B2. Perceptrón como clasificador booleano
Un perceptrón $y=H(w\cdot x+b)$. Buscar a mano pesos y sesgo para AND, OR y $x_1\wedge\neg x_2$ y verificar con la tabla de verdad.

In [ ]:
def perceptron(w,b):
    w=np.asarray(w,float)
    return lambda X: H(X@w+b)
X=np.array([[0,0],[0,1],[1,0],[1,1]])
AND=perceptron([1,1],-1.5)
print(AND(X))
# TODO: OR y x1 AND (NOT x2)

### B3. Algoritmo de aprendizaje del perceptrón
Regla: si $\hat y\ne y$, $w\leftarrow w+\eta (y-\hat y)\,x$, $b\leftarrow b+\eta(y-\hat y)$. Implementarla y entrenar sobre AND, OR y XOR, contando errores por época.

In [ ]:
def entrenar_perceptron(X,y,eta=1.0,epocas=50):
    w=np.zeros(X.shape[1]); b=0.0; hist=[]
    for ep in range(epocas):
        err=0
        for xi,yi in zip(X,y):
            yh=float(xi@w+b>=0)
            if yh!=yi:
                w+=eta*(yi-yh)*xi; b+=eta*(yi-yh); err+=1
        hist.append(err)
        if err==0: break
    return w,b,hist
for nombre,y in [('AND',[0,0,0,1]),('OR',[0,1,1,1]),('XOR',[0,1,1,0])]:
    w,b,h=entrenar_perceptron(X,np.array(y))
    print(nombre,w,b,'errores por epoca:',h[:8],'...' if len(h)>8 else '')

### B4. Verificación experimental del teorema de Novikoff
Generar datos linealmente separables en $\mathbb R^2$ con margen $\gamma$ respecto de un $w^*$ dado, correr el perceptrón (con $y_i\in\{\pm1\}$) y comparar el número de errores con la cota $(R/\gamma)^2$ para distintos márgenes.

In [ ]:
def datos_separables(n,gamma,rng):
    ws=np.array([np.cos(0.5),np.sin(0.5)])
    X=[];y=[]
    while len(X)<n:
        p=rng.uniform(-1,1,2); m=p@ws
        if abs(m)>=gamma: X.append(p); y.append(np.sign(m))
    return np.array(X),np.array(y),ws
def perceptron_pm(X,y):
    w=np.zeros(2); errores=0
    for _ in range(10000):
        malos=[i for i in range(len(y)) if y[i]*(X[i]@w)<=0]
        if not malos: break
        i=malos[0]; w+=y[i]*X[i]; errores+=1
    return w,errores
for g in [0.3,0.1,0.03]:
    X2,y2,ws=datos_separables(200,g,rng)
    R=np.linalg.norm(X2,axis=1).max()
    w,e=perceptron_pm(X2,y2)
    print(f'gamma={g}: errores={e}, cota (R/gamma)^2={(R/g)**2:.0f}')

**B4.1** ¿Cuán ajustada es la cota? ¿Cómo depende el número de errores del orden de los datos? Repetir con el orden permutado.

In [ ]:
# TODO

### B5. XOR con una capa oculta
Implementar la red de A5 con `perceptron` y verificar la tabla de verdad. Graficar la región de decisión de cada neurona oculta y de la salida en $[-0.5,1.5]^2$.

In [ ]:
# TODO